<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white;'>
    <h1 style='margin: 0; font-size: 36px; font-weight: bold;'>🔍 Dimensionality Reduction & Anomaly Detection</h1>
    <p style='margin: 10px 0 0 0; font-size: 18px; opacity: 0.9;'>Hands-On Session: PCA, UMAP & Isolation Forest for Fraud Detection</p>
</div>

---

## 📋 Session Overview

In this hands-on notebook, you'll apply **PCA**, **UMAP**, and **Isolation Forest** to real-world fraud detection scenarios. You'll learn to:

- ✅ Reduce high-dimensional transaction data using PCA while preserving variance
- ✅ Choose optimal number of components using explained variance and scree plots
- ✅ Visualize complex patterns using UMAP for non-linear dimensionality reduction
- ✅ Detect fraudulent transactions using Isolation Forest without labeled data
- ✅ Build an end-to-end fraud detection pipeline combining all techniques

**Estimated time:** 90-110 minutes

<div style='background-color: #e7f3ff; border-left: 5px solid #2196F3; padding: 15px; margin: 20px 0;'>
    <h3 style='margin-top: 0; color: #1976D2;'>🎯 Learning Goals</h3>
    <ol style='margin-bottom: 0;'>
        <li>Apply PCA for dimensionality reduction and understand variance maximization</li>
        <li>Interpret explained variance ratios to choose optimal component count</li>
        <li>Use UMAP as a fast, non-linear alternative for visualization</li>
        <li>Detect anomalies using Isolation Forest with contamination tuning</li>
        <li>Build a complete fraud detection pipeline from raw data to anomaly scores</li>
    </ol>
</div>

<div style='background-color: #4CAF50; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 1: Setup & Data Loading</h2>
</div>

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

# Install umap-learn if needed
try:
    import umap
except ImportError:
    print('Installing umap-learn...')
    !pip install umap-learn --break-system-packages
    import umap

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Libraries imported successfully')

### 📊 Dataset Introduction

We'll work with a **synthetic credit card transaction dataset** simulating real-world fraud detection:
- **30 features:** Transaction amount, merchant category codes, time features, user behavior metrics, velocity checks, device fingerprints, etc.
- **5,000 transactions** with approximately 5% fraudulent (unlabeled)
- **High dimensionality:** Patterns exist across all 30 dimensions, making manual inspection impossible

**Business objective:** Identify fraudulent transactions without labeled fraud examples using unsupervised anomaly detection.

In [ ]:
# Generate synthetic fraud detection dataset
np.random.seed(42)

# Create high-dimensional dataset with anomalies
n_samples = 5000
n_features = 30
n_fraud = int(n_samples * 0.05)  # 5% fraud

# Normal transactions (95%)
X_normal = np.random.randn(n_samples - n_fraud, n_features)

# Fraudulent transactions (5%) - shifted distribution
X_fraud = np.random.randn(n_fraud, n_features) * 1.5 + 2.0  # Different mean and variance

# Combine and create labels (for validation only - not used in training)
X = np.vstack([X_normal, X_fraud])
y_true = np.array([0] * (n_samples - n_fraud) + [1] * n_fraud)  # 0=normal, 1=fraud

# Shuffle
shuffle_idx = np.random.permutation(n_samples)
X = X[shuffle_idx]
y_true = y_true[shuffle_idx]

# Create DataFrame with feature names
feature_names = [f'feature_{i+1}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['true_label'] = y_true  # For evaluation only

print(f'✅ Generated {len(df)} transactions with {n_features} features')
print(f'True fraud rate: {y_true.sum() / len(y_true) * 100:.1f}%')
print(f'\nDataset shape: {df.shape}')
df.head()

<div style='background-color: #FF9800; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 2: Exploratory Data Analysis</h2>
</div>

In [ ]:
# Display summary statistics
print('📊 Dataset Summary Statistics:')
print(df[feature_names].describe())

print('\n📋 Data Info:')
print(df.info())

print('\n🔍 Missing Values:')
print(df.isnull().sum().sum())

In [ ]:
# Visualize feature correlations
plt.figure(figsize=(12, 10))
correlation_matrix = df[feature_names].corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 High correlation between some features suggests dimensionality reduction will be effective')

<div style='background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; margin: 20px 0;'>
    <h4 style='margin-top: 0; color: #856404;'>💡 Key Insight</h4>
    <p style='margin-bottom: 0;'>With 30 correlated features, PCA can likely compress this data significantly while preserving variance. The high dimensionality makes visualization impossible without reduction techniques.</p>
</div>

<div style='background-color: #9C27B0; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 3: Data Preprocessing</h2>
</div>

In [ ]:
# Separate features from true labels
X_features = df[feature_names].values

# Standardize features (critical for PCA and distance-based algorithms)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

print('✅ Features standardized (zero mean, unit variance)')
print(f'Original data shape: {X_features.shape}')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'\nScaled data mean per feature: {X_scaled.mean(axis=0)[:5]} ... (showing first 5)')
print(f'Scaled data std per feature: {X_scaled.std(axis=0)[:5]} ... (showing first 5)')

<div style='background-color: #ffebee; border-left: 5px solid #f44336; padding: 15px; margin: 20px 0;'>
    <h4 style='margin-top: 0; color: #c62828;'>⚠️ Critical Step</h4>
    <p style='margin-bottom: 0;'><strong>Why standardization matters:</strong> PCA is scale-sensitive. Features with larger scales would dominate variance calculations, making principal components reflect scale rather than actual patterns. Standardization ensures all features contribute equally.</p>
</div>

<div style='background-color: #2196F3; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 4: PCA - Principal Component Analysis</h2>
</div>

### 🔍 Step 1: Apply Full PCA to Understand Variance Distribution

In [ ]:
# Apply PCA with all components
pca_full = PCA()
pca_full.fit(X_scaled)

# Get explained variance ratio
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f'✅ Full PCA computed with {len(explained_variance)} components')
print(f'\nExplained variance by first 10 components:')
for i in range(min(10, len(explained_variance))):
    print(f'  PC{i+1}: {explained_variance[i]:.3f} ({explained_variance[i]*100:.1f}%)')

print(f'\nCumulative variance:')
print(f'  First 5 components: {cumulative_variance[4]:.3f} ({cumulative_variance[4]*100:.1f}%)')
print(f'  First 10 components: {cumulative_variance[9]:.3f} ({cumulative_variance[9]*100:.1f}%)')
print(f'  First 15 components: {cumulative_variance[14]:.3f} ({cumulative_variance[14]*100:.1f}%)')

### 📊 Step 2: Scree Plot - Finding the Elbow

In [ ]:
# Create scree plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Individual variance explained
axes[0].bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Variance Explained', fontsize=12)
axes[0].set_title('Scree Plot - Individual Variance', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative variance
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 
             marker='o', linewidth=2, markersize=6)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Variance Explained', fontsize=12)
axes[1].set_title('Cumulative Variance Explained', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Find n_components for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f'\n💡 Number of components needed for 95% variance: {n_components_95}')
print(f'💡 Compression ratio: {n_features} → {n_components_95} ({n_components_95/n_features*100:.1f}% of original)')

### 🎯 Step 3: Apply PCA with Optimal Components

In [ ]:
# Apply PCA with 95% variance threshold
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print(f'✅ PCA applied with variance threshold = 0.95')
print(f'Reduced from {X_scaled.shape[1]} to {X_pca.shape[1]} dimensions')
print(f'Total variance preserved: {pca.explained_variance_ratio_.sum():.3f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)')
print(f'\nComponent variance breakdown:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.3f} ({var*100:.1f}%)')

In [ ]:
# Visualize PCA loadings (feature contributions to components)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=feature_names
)

# Plot loadings for first 3 components
plt.figure(figsize=(14, 6))
loadings[['PC1', 'PC2', 'PC3']].plot(kind='bar', figsize=(14, 6))
plt.xlabel('Original Features', fontsize=12)
plt.ylabel('Loading Value', fontsize=12)
plt.title('PCA Loadings - Feature Contributions to Principal Components', fontsize=14, fontweight='bold')
plt.legend(title='Components')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('💡 Loadings show which original features contribute most to each principal component')

<div style='background-color: #E91E63; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 5: UMAP - Non-Linear Dimensionality Reduction</h2>
</div>

### 🔍 UMAP for Visualization

While PCA reduces to interpretable components, **UMAP** excels at 2D/3D visualization by preserving non-linear manifold structure. We'll reduce our data to 2D for visual exploration of anomalies.

In [ ]:
# Apply UMAP for 2D visualization
reducer = umap.UMAP(
    n_neighbors=15,      # Local structure: 15 neighbors
    min_dist=0.1,        # Minimum distance between points
    n_components=2,      # Reduce to 2D
    random_state=42
)

print('Computing UMAP embedding... (this may take 30-60 seconds)')
X_umap = reducer.fit_transform(X_scaled)

print(f'✅ UMAP applied: {X_scaled.shape} → {X_umap.shape}')
print(f'Reduced from {X_scaled.shape[1]}D to 2D for visualization')

In [ ]:
# Visualize UMAP embedding colored by true labels (for validation)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# UMAP colored by true fraud labels
scatter1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1], 
                           c=y_true, cmap='coolwarm', 
                           s=30, alpha=0.6, edgecolors='none')
axes[0].set_xlabel('UMAP 1', fontsize=12)
axes[0].set_ylabel('UMAP 2', fontsize=12)
axes[0].set_title('UMAP Projection (Colored by True Labels)', fontsize=14, fontweight='bold')
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('0=Normal, 1=Fraud', rotation=270, labelpad=20)

# PCA (2 components) for comparison
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                           c=y_true, cmap='coolwarm', 
                           s=30, alpha=0.6, edgecolors='none')
axes[1].set_xlabel('PC 1', fontsize=12)
axes[1].set_ylabel('PC 2', fontsize=12)
axes[1].set_title('PCA Projection (2D)', fontsize=14, fontweight='bold')
cbar2 = plt.colorbar(scatter2, ax=axes[1])
cbar2.set_label('0=Normal, 1=Fraud', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

print('💡 Notice how UMAP creates tighter, more separated clusters compared to PCA')

<div style='background-color: #00BCD4; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 6: Isolation Forest - Anomaly Detection</h2>
</div>

### 🎯 Unsupervised Fraud Detection

We'll apply **Isolation Forest** to the PCA-reduced data to detect anomalies (fraud) without using labels. The algorithm identifies points that are easy to isolate in random decision trees.

In [ ]:
# Apply Isolation Forest on PCA-reduced data
iso_forest = IsolationForest(
    contamination=0.05,      # Expected 5% anomalies
    max_samples=256,         # Subsample for efficiency
    random_state=42,
    n_estimators=100
)

# Fit and predict
anomaly_labels = iso_forest.fit_predict(X_pca)  # -1 = anomaly, 1 = normal
anomaly_scores = iso_forest.score_samples(X_pca)  # More negative = more anomalous

# Convert to binary (0=normal, 1=anomaly)
predicted_anomalies = (anomaly_labels == -1).astype(int)

print('✅ Isolation Forest applied to PCA-reduced data')
print(f'Contamination parameter: {iso_forest.contamination}')
print(f'\nResults:')
print(f'  Predicted anomalies: {predicted_anomalies.sum()} ({predicted_anomalies.sum()/len(predicted_anomalies)*100:.2f}%)')
print(f'  Predicted normal: {(1-predicted_anomalies).sum()} ({(1-predicted_anomalies).sum()/len(predicted_anomalies)*100:.2f}%)')

# Anomaly score distribution
print(f'\nAnomaly score statistics:')
print(f'  Min (most anomalous): {anomaly_scores.min():.3f}')
print(f'  Max (least anomalous): {anomaly_scores.max():.3f}')
print(f'  Mean: {anomaly_scores.mean():.3f}')
print(f'  Median: {np.median(anomaly_scores):.3f}')

In [ ]:
# Evaluate against true labels (not used during training)
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_recall_fscore_support

# Confusion matrix
cm = confusion_matrix(y_true, predicted_anomalies)
print('📊 Confusion Matrix:')
print(cm)
print(f'\n  True Negatives (correct normal): {cm[0,0]}')
print(f'  False Positives (false alarms): {cm[0,1]}')
print(f'  False Negatives (missed fraud): {cm[1,0]}')
print(f'  True Positives (caught fraud): {cm[1,1]}')

# Metrics
precision, recall, f1, _ = precision_recall_fscore_support(y_true, predicted_anomalies, average='binary')
print(f'\n📈 Performance Metrics:')
print(f'  Precision: {precision:.3f} (of flagged transactions, {precision*100:.1f}% are actually fraud)')
print(f'  Recall: {recall:.3f} (caught {recall*100:.1f}% of all fraud)')
print(f'  F1-Score: {f1:.3f}')

# ROC-AUC using anomaly scores
roc_auc = roc_auc_score(y_true, -anomaly_scores)  # Negate scores (lower=anomaly)
print(f'  ROC-AUC: {roc_auc:.3f}')

print('\n✅ Unsupervised detection achieved without seeing any labeled fraud examples!')

In [ ]:
# Visualize detected anomalies in UMAP space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# True labels
scatter1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1], 
                           c=y_true, cmap='coolwarm', 
                           s=30, alpha=0.6, edgecolors='none')
axes[0].set_xlabel('UMAP 1', fontsize=12)
axes[0].set_ylabel('UMAP 2', fontsize=12)
axes[0].set_title('Ground Truth (True Labels)', fontsize=14, fontweight='bold')
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('0=Normal, 1=Fraud', rotation=270, labelpad=20)

# Predicted anomalies
scatter2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], 
                           c=predicted_anomalies, cmap='coolwarm', 
                           s=30, alpha=0.6, edgecolors='none')
axes[1].set_xlabel('UMAP 1', fontsize=12)
axes[1].set_ylabel('UMAP 2', fontsize=12)
axes[1].set_title('Isolation Forest Predictions', fontsize=14, fontweight='bold')
cbar2 = plt.colorbar(scatter2, ax=axes[1])
cbar2.set_label('0=Normal, 1=Anomaly', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze anomaly score distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Score distribution by true label
axes[0].hist(-anomaly_scores[y_true == 0], bins=50, alpha=0.7, label='Normal (True)', color='blue')
axes[0].hist(-anomaly_scores[y_true == 1], bins=50, alpha=0.7, label='Fraud (True)', color='red')
axes[0].set_xlabel('Anomaly Score (higher = more anomalous)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Anomaly Score Distribution by True Label', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Top anomalies
top_n = 100
top_anomaly_idx = np.argsort(anomaly_scores)[:top_n]
top_anomaly_scores = -anomaly_scores[top_anomaly_idx]
top_true_labels = y_true[top_anomaly_idx]

axes[1].scatter(range(top_n), top_anomaly_scores, 
                c=top_true_labels, cmap='coolwarm', s=50, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Rank (most anomalous first)', fontsize=12)
axes[1].set_ylabel('Anomaly Score', fontsize=12)
axes[1].set_title(f'Top {top_n} Anomalies (Colored by True Label)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fraud_in_top_100 = top_true_labels.sum()
print(f'\n💡 Among top 100 most anomalous transactions: {fraud_in_top_100} are actually fraud ({fraud_in_top_100}%)')

<div style='background-color: #673AB7; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 7: Parameter Tuning & Experimentation</h2>
</div>

### 🧪 Experiment 1: Effect of Contamination Parameter

In [ ]:
# Test different contamination values
contamination_values = [0.01, 0.03, 0.05, 0.07, 0.10]
results = []

for cont in contamination_values:
    iso = IsolationForest(contamination=cont, random_state=42)
    preds = (iso.fit_predict(X_pca) == -1).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
    
    results.append({
        'contamination': cont,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'flagged': preds.sum()
    })

results_df = pd.DataFrame(results)
print('📊 Contamination Parameter Tuning Results:')
print(results_df.to_string(index=False))

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(results_df['contamination'], results_df['precision'], marker='o', label='Precision', linewidth=2)
axes[0].plot(results_df['contamination'], results_df['recall'], marker='s', label='Recall', linewidth=2)
axes[0].plot(results_df['contamination'], results_df['f1'], marker='^', label='F1-Score', linewidth=2)
axes[0].set_xlabel('Contamination Parameter', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Performance vs Contamination', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(results_df['contamination'], results_df['flagged'], alpha=0.7)
axes[1].axhline(y=y_true.sum(), color='red', linestyle='--', label=f'True fraud count ({y_true.sum()})')
axes[1].set_xlabel('Contamination Parameter', fontsize=12)
axes[1].set_ylabel('Number Flagged', fontsize=12)
axes[1].set_title('Transactions Flagged vs Contamination', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 🧪 Experiment 2: Effect of PCA Dimensionality

In [ ]:
# Test Isolation Forest on different PCA dimensions
n_components_list = [5, 10, 15, 20, 30]
pca_results = []

for n_comp in n_components_list:
    # Apply PCA
    pca_temp = PCA(n_components=min(n_comp, X_scaled.shape[1]))
    X_temp = pca_temp.fit_transform(X_scaled)
    
    # Apply Isolation Forest
    iso = IsolationForest(contamination=0.05, random_state=42)
    preds = (iso.fit_predict(X_temp) == -1).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
    
    pca_results.append({
        'n_components': n_comp,
        'variance_explained': pca_temp.explained_variance_ratio_.sum(),
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

pca_results_df = pd.DataFrame(pca_results)
print('📊 PCA Dimensionality Impact on Anomaly Detection:')
print(pca_results_df.to_string(index=False))

# Plot
plt.figure(figsize=(12, 6))
plt.plot(pca_results_df['n_components'], pca_results_df['f1'], marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of PCA Components', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.title('Anomaly Detection Performance vs PCA Dimensions', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Sweet spot balances compression (fewer components) with preserved signal (higher F1)')

<div style='background-color: #FF5722; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 8: End-to-End Fraud Detection Pipeline</h2>
</div>

### 🚀 Production-Ready Pipeline

Putting it all together: a complete pipeline from raw transactions to anomaly scores.

In [ ]:
# Complete fraud detection pipeline
class FraudDetectionPipeline:
    def __init__(self, n_components=0.95, contamination=0.05):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components)
        self.iso_forest = IsolationForest(contamination=contamination, random_state=42)
        
    def fit(self, X):
        """Train the pipeline on transaction data"""
        # Step 1: Standardize
        X_scaled = self.scaler.fit_transform(X)
        
        # Step 2: PCA dimensionality reduction
        X_pca = self.pca.fit_transform(X_scaled)
        
        # Step 3: Train anomaly detector
        self.iso_forest.fit(X_pca)
        
        print(f'✅ Pipeline trained')
        print(f'   PCA: {X.shape[1]} → {self.pca.n_components_} dimensions')
        print(f'   Variance preserved: {self.pca.explained_variance_ratio_.sum():.2%}')
        
        return self
    
    def predict(self, X):
        """Detect anomalies in new transactions"""
        X_scaled = self.scaler.transform(X)
        X_pca = self.pca.transform(X_scaled)
        anomaly_labels = self.iso_forest.predict(X_pca)
        return (anomaly_labels == -1).astype(int)
    
    def score(self, X):
        """Get continuous anomaly scores"""
        X_scaled = self.scaler.transform(X)
        X_pca = self.pca.transform(X_scaled)
        return -self.iso_forest.score_samples(X_pca)  # Flip: higher = more anomalous

# Train pipeline
pipeline = FraudDetectionPipeline(n_components=0.95, contamination=0.05)
pipeline.fit(X_features)

# Get predictions and scores
predictions = pipeline.predict(X_features)
risk_scores = pipeline.score(X_features)

print(f'\n📊 Pipeline Results:')
print(f'   Flagged as fraud: {predictions.sum()} ({predictions.sum()/len(predictions)*100:.2f}%)')
print(f'   Risk score range: [{risk_scores.min():.3f}, {risk_scores.max():.3f}]')

In [ ]:
# Create fraud report for business stakeholders
fraud_report = pd.DataFrame({
    'transaction_id': range(len(X_features)),
    'risk_score': risk_scores,
    'is_flagged': predictions,
    'true_fraud': y_true  # In production, this wouldn't be available
})

# Sort by risk score
fraud_report = fraud_report.sort_values('risk_score', ascending=False)

print('🚨 Top 10 Highest Risk Transactions:')
print(fraud_report.head(10).to_string(index=False))

print('\n✅ Low Risk Sample (Normal Transactions):')
print(fraud_report.tail(5).to_string(index=False))

<div style='background-color: #607D8B; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 9: Summary & Key Takeaways</h2>
</div>

<div style='background-color: #e8f5e9; border-left: 5px solid #4CAF50; padding: 20px; margin: 20px 0;'>
    <h3 style='margin-top: 0; color: #2E7D32;'>✅ What You've Learned</h3>
    
**PCA (Principal Component Analysis):**
- Reduces dimensionality by finding directions of maximum variance
- Explained variance ratio guides component selection (typically 90-95% threshold)
- Scree plots visualize variance distribution across components
- Always standardize features before PCA (scale-sensitive)
    
**UMAP (Uniform Manifold Approximation and Projection):**
- Non-linear dimensionality reduction preserving local and global structure
- Excels at visualization of complex, high-dimensional data
- n_neighbors controls local vs global balance
- Stochastic (use random_state for reproducibility)
    
**Isolation Forest:**
- Unsupervised anomaly detection via random tree isolation
- Contamination parameter sets expected anomaly proportion
- Anomalies have short path lengths (easy to isolate)
- Use score_samples() for continuous risk scores
    
**Pipeline Integration:**
- PCA first (compression), then Isolation Forest (detection), then UMAP (visualization)
- Validate performance with held-out labels when available
- Tune contamination based on business requirements (investigation capacity)
</div>

### 🎯 Decision Framework: When to Use Each Technique

**Use PCA when:**
- ✅ Need interpretable components (loadings show feature contributions)
- ✅ Relationships are primarily linear
- ✅ Speed is critical (PCA is 10-100x faster than UMAP)
- ✅ Want variance-based feature reduction for downstream models

**Use UMAP when:**
- ✅ Visualizing complex, non-linear patterns
- ✅ Data lies on curved manifolds (circular, spiral patterns)
- ✅ Want to preserve both local neighborhoods and global structure
- ✅ Okay with non-interpretable components

**Use Isolation Forest when:**
- ✅ No labeled anomaly examples available
- ✅ Anomalies are rare and different from normal data
- ✅ Need fast, scalable anomaly detection
- ✅ High-dimensional data (works well with 100+ features)

**Combine all three when:**
- ✅ Building production fraud detection systems
- ✅ Need both detection (Isolation Forest) and explainability (UMAP visualization)
- ✅ Data is high-dimensional and patterns are complex

### 🚀 Next Steps

1. Apply these techniques to your own high-dimensional datasets
2. Explore other dimensionality reduction methods (t-SNE, Autoencoders)
3. Learn about other anomaly detectors (Local Outlier Factor, One-Class SVM)
4. Practice tuning contamination based on business constraints
5. Combine with supervised models when labels become available

---

<div style='text-align: center; padding: 20px; background-color: #f5f5f5; border-radius: 10px;'>
    <h3 style='color: #333;'>🎓 Session Complete!</h3>
    <p style='color: #666;'>You've successfully built an end-to-end fraud detection pipeline using PCA, UMAP, and Isolation Forest.</p>
</div>